In [ ]:
!pip install transformers datasets torch scikit-learn -q

In [ ]:
import pandas as pd

# Load from HuggingFace directly — no script issues
from datasets import load_dataset
dataset = load_dataset("GonzaloA/fake_news")

train_df = pd.DataFrame(dataset['train'])
test_df  = pd.DataFrame(dataset['test'])
val_df   = pd.DataFrame(dataset['validation'])

print("Train size:", len(train_df))
print("Columns:", train_df.columns.tolist())
print("Label distribution:\n", train_df['label'].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/38.8M [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/13.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24353 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8117 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8117 [00:00<?, ? examples/s]

Train size: 24353
Columns: ['Unnamed: 0', 'title', 'text', 'label']
Label distribution:
 label
1    13195
0    11158
Name: count, dtype: int64


In [ ]:
train_df = train_df[['text', 'label']].dropna()
test_df  = test_df[['text', 'label']].dropna()
val_df   = val_df[['text', 'label']].dropna()

# Rename to match our pipeline
train_df.columns = ['statement', 'binary_label']
test_df.columns  = ['statement', 'binary_label']
val_df.columns   = ['statement', 'binary_label']

# Truncate to 256 chars (articles are long, we keep it manageable)
train_df['statement'] = train_df['statement'].str[:512]
test_df['statement']  = test_df['statement'].str[:512]
val_df['statement']   = val_df['statement'].str[:512]

print("Train:", len(train_df))
print("Label dist:\n", train_df['binary_label'].value_counts())

train_df.to_csv('train.csv', index=False)
test_df.to_csv('test.csv', index=False)
val_df.to_csv('val.csv', index=False)
print("✅ Saved CSVs")

Train: 24353
Label dist:
 binary_label
1    13195
0    11158
Name: count, dtype: int64
✅ Saved CSVs


In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN    = 256   # longer because full articles now

class NewsDataset(Dataset):
    def __init__(self, df):
        self.texts  = df['statement'].tolist()
        self.labels = df['binary_label'].tolist()

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = NewsDataset(train_df)
val_dataset   = NewsDataset(val_df)
test_dataset  = NewsDataset(test_df)
print("Datasets ready!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Datasets ready!


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0:"FAKE", 1:"REAL"},
    label2id={"FAKE":0, "REAL":1}
)
model.to(device)
print("Model ready!")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model ready!


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average='weighted')
    }

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    save_total_limit=1,
    fp16=True,
    report_to='none'
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 Training...")
trainer.train()

🚀 Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.032365,0.036202,0.987187,0.987190
2,0.020708,0.032642,0.989775,0.989776
3,0.005124,0.045790,0.990391,0.990392
4,0.005055,0.049271,0.989898,0.989899


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3048, training_loss=0.033760766673520244, metrics={'train_runtime': 783.6215, 'train_samples_per_second': 124.31, 'train_steps_per_second': 3.89, 'total_flos': 6451957118939136.0, 'train_loss': 0.033760766673520244, 'epoch': 4.0})

In [ ]:
# Cell 11 — Final evaluation
results = trainer.evaluate(test_dataset)
print(f"\n✅ Test Accuracy: {results['eval_accuracy']*100:.1f}%")
print(f"   F1 Score: {results['eval_f1']:.4f}")


✅ Test Accuracy: 98.9%
   F1 Score: 0.9887


In [ ]:
# Cell 12 — Save model
model.save_pretrained('./saved_model')
tokenizer.save_pretrained('./saved_model')
print("✅ Saved to ./saved_model/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Saved to ./saved_model/


In [ ]:
from google.colab import files
import shutil

shutil.make_archive('saved_model', 'zip', './saved_model')
files.download('saved_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import os
print(os.listdir('./saved_model'))

['model.safetensors', 'config.json', 'tokenizer.json', 'tokenizer_config.json']


In [18]:
from google.colab import files

for filename in os.listdir('./saved_model'):
    print(f"Downloading {filename}...")
    files.download(f'./saved_model/{filename}')

print("✅ All files downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All files downloaded!
